# 阶段八：接入真实 I/O

使用 `httpx.AsyncClient` 演示客户端复用、请求超时和取消清理。MockTransport 只用于让实验不依赖外网。


In [ ]:
import asyncio
import httpx


async def mock_handler(request: httpx.Request) -> httpx.Response:
    """模拟 HTTP 服务。

    参数:
        request: httpx 发出的请求对象。

    返回:
        包含请求路径的 JSON 响应。
    """
    await asyncio.sleep(0.01)
    return httpx.Response(200, json={"path": request.url.path})


async def fetch_json(client: httpx.AsyncClient, path: str, timeout: float = 1.0) -> dict[str, str]:
    """使用已有客户端请求 JSON。

    参数:
        client: 可复用的异步 HTTP 客户端。
        path: 请求路径，例如 `/weather`。
        timeout: 本次请求允许等待的秒数。

    返回:
        解析后的 JSON 字典。
    """
    response = await client.get(path, timeout=timeout)
    response.raise_for_status()
    return response.json()


async def run_http_tools(paths: list[str]) -> list[dict[str, str]]:
    """复用一个客户端并发执行多个 HTTP 工具。

    参数:
        paths: 要请求的路径列表。

    返回:
        与 paths 顺序一致的 JSON 结果列表。
    """
    transport = httpx.MockTransport(mock_handler)
    async with httpx.AsyncClient(base_url="https://example.test", transport=transport) as client:
        return await asyncio.gather(*(fetch_json(client, path) for path in paths))


async def cancel_http_request() -> None:
    """取消一个 HTTP 工具任务，验证取消继续传播。

    该方法不接收参数；请求任务被取消后，`async with` 负责关闭客户端。
    """
    transport = httpx.MockTransport(mock_handler)
    async with httpx.AsyncClient(base_url="https://example.test", transport=transport) as client:
        task = asyncio.create_task(fetch_json(client, "/cancelled"))
        task.cancel()
        try:
            await task
        except asyncio.CancelledError:
            print("HTTP task cancelled and client cleanup completed")


async def main() -> None:
    """验证异步 HTTP 请求、客户端复用和取消清理。"""
    results = await run_http_tools(["/weather", "/search"])
    assert results == [{"path": "/weather"}, {"path": "/search"}]
    print("shared AsyncClient results:", results)
    await cancel_http_request()


await main()
